# LOB Deep Learning - Implementation Code

**Hierarchical 2D Attention with Multi-Task Learning for Financial Time-Series Direction Prediction**

AI Methods - Rutgers Business School - Spring 2026

---

This notebook contains the full source code for the project, organised as runnable cells with explanatory markdown. Use `showcase_colab.ipynb` for results and visualisations.


## Table of Contents

1. **Configuration** - `config.py`
2. **Data Pipeline**
    - 2a. Streaming JSON -> Parquet preprocessing - `data/preprocess.py`
    - 2b. PyTorch Dataset & walk-forward split - `data/dataset.py`
3. **Model Architectures**
    - 3a/3b. Logistic Regression + MLP - `models/baseline.py`
    - 3c. CNN + LSTM - `models/cnn_lstm.py`
    - 3d. DeepLOB - `models/deeplob.py`
    - 3e. LOBTransformer (our contribution) - `models/lob_transformer.py`
    - 3f. BiGRU + Attention - `models/bigru_attention.py`
4. **Training Loop** - `train.py`
5. **Evaluation** - `evaluate.py`
6. **Backtest** - `backtest.py`
7. **CLI Driver** - `run.py`
8. **Utilities** - `utils.py`
9. **Innovation Code** - FGSM + animated attention


## 1. Configuration (`config.py`)

Central dataclass holding all hyperparameters and paths.


In [9]:
from dataclasses import dataclass, field
from typing import List
import os

# Updated for Colab environment.
# If using Google Drive, mount it and change this to '/content/drive/MyDrive/...'
BASE = "/content"

@dataclass
class Config:
    # ── Paths ────────────────────────────────────────────────────────────────
    data_root:     str = os.path.join(BASE, "Data", "LTC", "ALOBdata", "CryptoLOB-2025")
    processed_dir: str = os.path.join(BASE, "LOB_DeepLearning", "processed")
    save_dir:      str = os.path.join(BASE, "LOB_DeepLearning", "checkpoints")
    results_dir:   str = os.path.join(BASE, "LOB_DeepLearning", "results")

    # ── Data ─────────────────────────────────────────────────────────────────
    symbols:   List[str] = field(default_factory=lambda: ["BTC", "ETH"])
    n_levels:  int   = 10      # top N bid/ask levels to use (50 available)

    # ── Sequence / Labels ────────────────────────────────────────────────────
    seq_len:   int         = 100          # time steps of history per sample
    horizons:  List[int]   = field(default_factory=lambda: [10, 20, 50])
    alpha:     float       = 0.001        # threshold: |Deltamid/mid| > alpha -> up/down
    n_smooth:  int         = 5            # future ticks to average for label smoothing

    # ── Training ─────────────────────────────────────────────────────────────
    batch_size:   int   = 64
    lr:           float = 1e-4
    weight_decay: float = 1e-4
    epochs:       int   = 50
    patience:     int   = 10             # early stopping patience
    n_folds:      int   = 5              # walk-forward CV folds

    # ── Model ────────────────────────────────────────────────────────────────
    # choices: "lr" | "mlp" | "cnn_lstm" | "deeplob" | "lob_transformer"
    model_name:    str   = "lob_transformer"
    horizon_key:   int   = 10            # which horizon to train on (10 | 20 | 50)
    d_level:       int   = 32            # level encoder dim (LOBTransformer)
    d_model:       int   = 128           # temporal transformer dim
    n_heads:       int   = 8
    n_layers:      int   = 4
    dropout:       float = 0.1
    lstm_hidden:   int   = 128           # CNN+LSTM / DeepLOB hidden size

    # ── Backtest ─────────────────────────────────────────────────────────────
    transaction_cost: float = 0.0005     # 0.05% per side (Binance taker)

    # ── Misc ─────────────────────────────────────────────────────────────────
    device:        str   = "cuda"        # "cuda" or "cpu"
    num_workers:   int   = 0
    seed:          int   = 42
    debug:         bool  = False         # if True, process/train on tiny subset

    def __post_init__(self):
        os.makedirs(self.processed_dir, exist_ok=True)
        os.makedirs(self.save_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)

        import torch
        if self.device == "cuda" and not torch.cuda.is_available():
            self.device = "cpu"

    @property
    def n_features(self) -> int:
        return self.n_levels * 4   # bid_p, bid_v, ask_p, ask_v per level

    @property
    def label_col(self) -> str:
        return f"label_k{self.horizon_key}"

## 2a. Streaming JSON -> Parquet (`data/preprocess.py`)

Memory-bounded ingestion of the 10 GB raw nested-JSON dataset using `ijson`. Outputs daily Parquet files with normalized features and quantile-balanced labels.


In [10]:
!pip install ijson

In [11]:
"""
Streaming JSON → daily Parquet preprocessing pipeline.

Run once per symbol:
    python -m data.preprocess --symbol BTC
    python -m data.preprocess --symbol ETH

Output: processed/<SYMBOL>/<YYYY-MM-DD>.parquet
Each row = one LOB snapshot with normalised features + labels for k={10,20,50}.
"""

import os
import math
import argparse
from datetime import datetime, timezone
from collections import defaultdict
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import ijson
from tqdm import tqdm

# ── helpers ──────────────────────────────────────────────────────────────

def _parse_ts(ts_str: str) -> datetime:
    ts_str = ts_str.replace("Z", "+00:00")
    try:
        return datetime.fromisoformat(ts_str)
    except ValueError:
        # truncate sub-microsecond precision if needed
        parts = ts_str.split(".")
        if len(parts) == 2:
            frac, tz = parts[1][:6], parts[1][6:]
            ts_str = parts[0] + "." + frac + tz
        return datetime.fromisoformat(ts_str)


def _extract_levels(snapshot: Dict[str, Any], n_levels: int):
    """Return (bid_prices, bid_sizes, ask_prices, ask_sizes) as float32 arrays."""
    bids = snapshot["bids"][:n_levels]
    asks = snapshot["asks"][:n_levels]

    # pad if fewer levels than n_levels (rare edge case)
    while len(bids) < n_levels:
        bids.append({"price": float(bids[-1]["price"]), "size": 0.0})
    while len(asks) < n_levels:
        asks.append({"price": float(asks[-1]["price"]), "size": 0.0})

    bp = np.array([float(b["price"]) for b in bids], dtype=np.float64)
    bv = np.array([float(b["size"])  for b in bids], dtype=np.float64)
    ap = np.array([float(a["price"]) for a in asks], dtype=np.float64)
    av = np.array([float(a["size"])  for a in asks], dtype=np.float64)
    return bp, bv, ap, av


def _compute_ofi(bp, bv, ap, av,
                 prev_bp, prev_bv, prev_ap, prev_av, n_levels: int):
    """
    Order Flow Imbalance per level.
    OFI_i = delta_bid_volume_i - delta_ask_volume_i, where delta accounts for
    price changes (new level → full volume; removed level → negative full volume).
    Returns array of shape [n_levels].
    """
    ofi = np.zeros(n_levels, dtype=np.float32)
    if prev_bp is None:
        return ofi
    for i in range(n_levels):
        # bid side
        if bp[i] > prev_bp[i]:
            d_bid = bv[i]
        elif bp[i] == prev_bp[i]:
            d_bid = bv[i] - prev_bv[i]
        else:
            d_bid = -prev_bv[i]
        # ask side
        if ap[i] < prev_ap[i]:
            d_ask = av[i]
        elif ap[i] == prev_ap[i]:
            d_ask = av[i] - prev_av[i]
        else:
            d_ask = -prev_av[i]
        ofi[i] = float(d_bid - d_ask)
    return ofi


def _compute_labels(mid_prices: np.ndarray, horizons: List[int],
                    alpha: float, n_smooth: int) -> Dict[str, np.ndarray]:
    """
    Quantile-based balanced label generation (recommended over fixed alpha).
    For each horizon k:
      1. Compute smoothed future return: pct_t = (mean(mid[t+1..t+k]) - mid[t]) / mid[t]
      2. Threshold at 33rd / 67th percentile -> equal-sized UP/STAT/DOWN classes
    Labels: 0=DOWN, 1=STATIONARY, 2=UP
    """
    T = len(mid_prices)
    labels = {}
    for k in horizons:
        pcts = np.zeros(T, dtype=np.float32)
        for t in range(T - k):
            future = mid_prices[t + 1 : t + 1 + min(n_smooth, k)]
            if len(future) == 0:
                continue
            m_now = mid_prices[t]
            if m_now == 0:
                continue
            pcts[t] = (future.mean() - m_now) / m_now

        valid = pcts[:T - k]
        q33, q67 = np.percentile(valid, [33.3, 66.7])
        lbl = np.ones(T, dtype=np.int8)
        lbl[pcts < q33]  = 0   # DOWN
        lbl[pcts > q67]  = 2   # UP
        lbl[T - k:]      = 1   # no valid label at end

        up   = (lbl == 2).sum()
        down = (lbl == 0).sum()
        stat = (lbl == 1).sum()
        print(f"    k={k}: DOWN={down:,} ({down/T*100:.1f}%)  "
              f"STAT={stat:,} ({stat/T*100:.1f}%)  UP={up:,} ({up/T*100:.1f}%)")
        labels[f"label_k{k}"] = lbl
    return labels


def _compute_vol_label(mid_prices: np.ndarray, horizon: int = 50) -> np.ndarray:
    """Realised volatility bin (0=low, 1=med, 2=high) over next `horizon` ticks."""
    T = len(mid_prices)
    vol = np.zeros(T, dtype=np.float32)
    for t in range(T - horizon):
        slice_ = mid_prices[t : t + horizon]
        log_rets = np.diff(np.log(slice_ + 1e-12))
        vol[t] = float(log_rets.std())
    # bin into terciles using non-zero values
    nonzero = vol[vol > 0]
    if len(nonzero) < 3:
        return np.zeros(T, dtype=np.int8)
    q1, q2 = np.percentile(nonzero, [33.3, 66.7])
    bins = np.zeros(T, dtype=np.int8)
    bins[vol >= q1] = 1
    bins[vol >= q2] = 2
    return bins


# ── main per-file processor ──────────────────────────────────────────────────

def preprocess_file(json_path: str, out_dir: str, n_levels: int,
                    horizons: List[int], alpha: float, n_smooth: int,
                    debug: bool = False):
    """
    Stream one JSON file, group by calendar day (UTC), write one Parquet per day.
    """
    os.makedirs(out_dir, exist_ok=True)

    # buffers keyed by date string "YYYY-MM-DD"
    day_rows: Dict[str, list] = defaultdict(list)

    prev_bp = prev_bv = prev_ap = prev_av = None
    n_processed = 0
    debug_limit = 50000  # ~14 hours at 1Hz

    print(f"Streaming {os.path.basename(json_path)} ...")
    with open(json_path, "rb") as f:
        try:
            for snapshot in tqdm(ijson.items(f, "item.item"), unit=" snaps"):
                ts = _parse_ts(snapshot["time_exchange"])
                date_str = ts.strftime("%Y-%m-%d")

                bp, bv, ap, av = _extract_levels(snapshot, n_levels)

                best_bid = float(bp[0])
                best_ask = float(ap[0])
                if best_ask <= best_bid:
                    prev_bp, prev_bv, prev_ap, prev_av = bp, bv, ap, av
                    continue

                mid_price = (best_bid + best_ask) / 2.0
                spread    = best_ask - best_bid

                bp_norm = (mid_price - bp) / mid_price
                ap_norm = (ap - mid_price) / mid_price

                bv_log = np.log1p(bv).astype(np.float32)
                av_log = np.log1p(av).astype(np.float32)

                ofi = _compute_ofi(bp, bv, ap, av, prev_bp, prev_bv, prev_ap, prev_av, n_levels)

                row = {
                    "timestamp":  ts,
                    "mid_price":  mid_price,
                    "best_bid":   best_bid,
                    "best_ask":   best_ask,
                    "spread":     spread,
                }
                for i in range(n_levels):
                    row[f"bid_p{i}"] = float(bp_norm[i])
                    row[f"bid_v{i}"] = float(bv_log[i])
                    row[f"ask_p{i}"] = float(ap_norm[i])
                    row[f"ask_v{i}"] = float(av_log[i])
                    row[f"ofi_{i}"]  = float(ofi[i])

                day_rows[date_str].append(row)

                prev_bp, prev_bv, prev_ap, prev_av = bp, bv, ap, av
                n_processed += 1
                if debug and n_processed >= debug_limit:
                    print(f"[debug] stopping early at {n_processed} snapshots")
                    break
        except ijson.common.IncompleteJSONError as e:
            print(f"[warning] Truncated/concatenated JSON at snapshot {n_processed}: {e}")
            print(f"[warning] Continuing with {n_processed} snapshots already read.")

    # ── label generation + write per day ──────────────────────────────────
    print(f"Computing labels and writing Parquet files ...")
    for date_str, rows in sorted(day_rows.items()):
        df = pd.DataFrame(rows)
        mid = df["mid_price"].to_numpy()

        lbls = _compute_labels(mid, horizons, alpha, n_smooth)
        for col, arr in lbls.items():
            df[col] = arr

        df["vol_label"] = _compute_vol_label(mid)

        out_path = os.path.join(out_dir, f"{date_str}.parquet")
        df.to_parquet(out_path, index=False)
        print(f"  wrote {out_path}  ({len(df):,} rows)")

    print(f"Done. {n_processed:,} snapshots -> {len(day_rows)} daily files.\n")


# ── CLI ─────────────────────────────────────────────────────────────

def preprocess_symbol(symbol: str, config, debug: bool = False):
    symbol = symbol.upper()
    src_dir = os.path.join(config.data_root, symbol)
    out_dir = os.path.join(config.processed_dir, symbol)

    json_files = sorted(
        [os.path.join(src_dir, f) for f in os.listdir(src_dir) if f.endswith(".json")]
    )
    if not json_files:
        raise FileNotFoundError(f"No JSON files found in {src_dir}")

    print(f"\n{'='*60}")
    print(f"Preprocessing {symbol}: {len(json_files)} file(s)")
    print(f"Output: {out_dir}")
    print(f"{'='*60}\n")

    for jf in json_files:
        preprocess_file(
            json_path=jf,
            out_dir=out_dir,
            n_levels=config.n_levels,
            horizons=config.horizons,
            alpha=config.alpha,
            n_smooth=config.n_smooth,
            debug=debug,
        )
        if debug:
            print("[debug] stopping after first file")
            break


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--symbol", required=True, choices=["BTC", "ETH"])
    parser.add_argument("--debug", action="store_true")
    # In Jupyter notebooks, sys.argv includes the kernel arguments.
    # Pass a default argument list to avoid ArgumentError.
    args = parser.parse_args(args=["--symbol", "BTC"])

    import sys
    sys.path.insert(0, os.path.join(os.getcwd(), ".."))
    cfg = Config(debug=args.debug)
    try:
        preprocess_symbol(args.symbol, cfg, debug=args.debug)
    except FileNotFoundError as e:
        print(e)


[Errno 2] No such file or directory: '/content/Data/LTC/ALOBdata/CryptoLOB-2025/BTC'


## 2b. PyTorch Dataset & Walk-Forward Split (`data/dataset.py`)

Sliding-window tensors `[T=100, N=10, 4]` and the 70/15/15 chronological split. Z-score normalizer fit on train rows only.


In [12]:
"""
PyTorch Dataset and DataLoader factories for the LOB project.

Each sample:
    X     : float32 tensor [seq_len, n_levels, 4]
             channels per level = (bid_p, bid_v, ask_p, ask_v)
    y_dir : int64 scalar  (0=DOWN, 1=STATIONARY, 2=UP)
    y_vol : int64 scalar  (0=low, 1=med, 2=high volatility)

Walk-forward split (chronological, no data leakage):
    train  → first 70% of windows
    val    → next  15%
    test   → final 15%
"""

import os
from typing import Tuple, Optional, List

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


# ── column helpers ─────────────────────────────────────────────────────────────

def _feature_cols(n_levels: int) -> List[str]:
    cols = []
    for i in range(n_levels):
        cols += [f"bid_p{i}", f"bid_v{i}", f"ask_p{i}", f"ask_v{i}"]
    return cols


def _ofi_cols(n_levels: int) -> List[str]:
    return [f"ofi_{i}" for i in range(n_levels)]


# ── normalisation ──────────────────────────────────────────────────────────────

class FeatureNormalizer:
    """Z-score normaliser – fit on training data only, applied to all splits."""

    def __init__(self):
        self.mean: Optional[np.ndarray] = None
        self.std:  Optional[np.ndarray] = None

    def fit(self, X: np.ndarray):
        self.mean = X.mean(axis=0, keepdims=True)
        self.std  = X.std(axis=0,  keepdims=True).clip(min=1e-8)

    def transform(self, X: np.ndarray) -> np.ndarray:
        return ((X - self.mean) / self.std).astype(np.float32)

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        self.fit(X)
        return self.transform(X)


# ── dataset ────────────────────────────────────────────────────────────────────

class LOBDataset(Dataset):
    """
    Sliding-window dataset over preprocessed Parquet files.

    __getitem__ returns (X, y_dir, y_vol):
        X     : [seq_len, n_levels, 4]
        y_dir : int class (0/1/2 = down/stat/up)
        y_vol : int class (0/1/2 = low/med/high vol)
    """

    def __init__(
        self,
        parquet_dir: str,
        seq_len: int,
        label_col: str,
        n_levels: int,
        normalizer: Optional[FeatureNormalizer] = None,
        indices: Optional[np.ndarray] = None,
    ):
        self.seq_len   = seq_len
        self.label_col = label_col
        self.n_levels  = n_levels
        self.feat_cols = _feature_cols(n_levels)

        parquet_files = sorted(
            [os.path.join(parquet_dir, f)
             for f in os.listdir(parquet_dir) if f.endswith(".parquet")]
        )
        if not parquet_files:
            raise FileNotFoundError(f"No parquet files in {parquet_dir}")

        dfs = [pd.read_parquet(p) for p in parquet_files]
        df  = pd.concat(dfs, ignore_index=True)

        X_raw = df[self.feat_cols].to_numpy(dtype=np.float32)

        if normalizer is not None:
            X_raw = normalizer.transform(X_raw)

        self.X     = X_raw                                    # [T, 4*N]
        self.y_dir = df[label_col].to_numpy(dtype=np.int64)  # [T]

        # vol_label may not exist if preprocessed before this field was added
        if "vol_label" in df.columns:
            self.y_vol = df["vol_label"].to_numpy(dtype=np.int64)
        else:
            self.y_vol = np.zeros(len(df), dtype=np.int64)

        self.meta = df[["timestamp", "mid_price",
                         "best_bid", "best_ask"]].copy()

        max_start = len(self.X) - seq_len
        if max_start <= 0:
            raise ValueError(
                f"Not enough data: {len(self.X)} rows < seq_len={seq_len}"
            )

        self.indices = np.arange(max_start) if indices is None else indices

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        start = int(self.indices[idx])
        end   = start + self.seq_len
        x     = self.X[start:end].reshape(self.seq_len, self.n_levels, 4)
        y_dir = int(self.y_dir[end - 1])
        y_vol = int(self.y_vol[end - 1])
        return (torch.from_numpy(x.copy()),
                torch.tensor(y_dir, dtype=torch.long),
                torch.tensor(y_vol, dtype=torch.long))

    def get_meta(self, idx: int) -> pd.Series:
        start = int(self.indices[idx])
        return self.meta.iloc[start + self.seq_len - 1]


# ── OFI flat-feature dataset for baselines ─────────────────────────────────────

class OFIDataset(Dataset):
    """
    Hand-crafted OFI feature dataset for LR / MLP baselines.
    Returns (x_flat, y_dir, y_vol).
    """

    def __init__(
        self,
        parquet_dir: str,
        label_col: str,
        n_levels: int,
        normalizer: Optional[FeatureNormalizer] = None,
        indices: Optional[np.ndarray] = None,
    ):
        parquet_files = sorted(
            [os.path.join(parquet_dir, f)
             for f in os.listdir(parquet_dir) if f.endswith(".parquet")]
        )
        dfs = [pd.read_parquet(p) for p in parquet_files]
        df  = pd.concat(dfs, ignore_index=True)

        ofi_cols  = _ofi_cols(n_levels)
        bid_v_cols = [f"bid_v{i}" for i in range(n_levels)]
        ask_v_cols = [f"ask_v{i}" for i in range(n_levels)]

        ofi      = df[ofi_cols].to_numpy(dtype=np.float32)
        spread   = (df["spread"] / df["mid_price"]).to_numpy(dtype=np.float32).reshape(-1, 1)
        bid_v    = df[bid_v_cols].to_numpy(dtype=np.float32).sum(axis=1, keepdims=True)
        ask_v    = df[ask_v_cols].to_numpy(dtype=np.float32).sum(axis=1, keepdims=True)
        depth_imb = (bid_v - ask_v) / (bid_v + ask_v + 1e-8)

        X_raw = np.hstack([ofi, spread, depth_imb]).astype(np.float32)

        if normalizer is not None:
            X_raw = normalizer.transform(X_raw)

        self.X     = X_raw
        self.y_dir = df[label_col].to_numpy(dtype=np.int64)
        self.y_vol = (df["vol_label"].to_numpy(dtype=np.int64)
                      if "vol_label" in df.columns
                      else np.zeros(len(df), dtype=np.int64))

        self.indices = np.arange(len(self.X)) if indices is None else indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = int(self.indices[idx])
        return (torch.from_numpy(self.X[i].copy()),
                torch.tensor(int(self.y_dir[i]), dtype=torch.long),
                torch.tensor(int(self.y_vol[i]), dtype=torch.long))


# ── walk-forward split ─────────────────────────────────────────────────────────

def walk_forward_split(
    parquet_dir: str, seq_len: int,
    train_frac: float = 0.70, val_frac: float = 0.15
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Chronological split → (train_idx, val_idx, test_idx)."""
    parquet_files = sorted(
        [os.path.join(parquet_dir, f)
         for f in os.listdir(parquet_dir) if f.endswith(".parquet")]
    )
    total     = sum(len(pd.read_parquet(p)) for p in parquet_files)
    max_start = total - seq_len

    n_train = int(max_start * train_frac)
    n_val   = int(max_start * val_frac)

    return (np.arange(0, n_train),
            np.arange(n_train, n_train + n_val),
            np.arange(n_train + n_val, max_start))


# ── DataLoader factory ─────────────────────────────────────────────────────────

def get_dataloaders(config, symbol: str = "BTC"):
    """Build train/val/test DataLoaders + fitted normalizer."""
    parquet_dir = os.path.join(config.processed_dir, symbol)
    train_idx, val_idx, test_idx = walk_forward_split(parquet_dir, config.seq_len)

    # fit normalizer on training rows only
    normalizer   = FeatureNormalizer()
    raw_ds       = LOBDataset(parquet_dir, config.seq_len, config.label_col,
                               config.n_levels, normalizer=None, indices=train_idx)
    normalizer.fit(raw_ds.X[train_idx])

    def _make(idx, shuffle):
        ds = LOBDataset(parquet_dir, config.seq_len, config.label_col,
                        config.n_levels, normalizer=normalizer, indices=idx)
        return DataLoader(ds, batch_size=config.batch_size,
                          shuffle=shuffle, num_workers=config.num_workers,
                          pin_memory=(config.device == "cuda"))

    return (
        _make(train_idx, shuffle=True),
        _make(val_idx,   shuffle=False),
        _make(test_idx,  shuffle=False),
        normalizer,
    )


## 3a/3b. Classical Baselines (`models/baseline.py`)

**LogisticRegressionLOB** - sklearn linear model with balanced class weights.

**MLPLOB** - 3-layer PyTorch MLP `[12 -> 64 -> 32 -> 3]`.


In [13]:
"""
Classical baselines:
  - LogisticRegressionLOB  (sklearn, uses OFI flat features)
  - MLPLOB                 (3-layer PyTorch MLP, uses OFI flat features)
"""

import numpy as np
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


class LogisticRegressionLOB:
    """Thin sklearn wrapper with balanced class weights."""

    def __init__(self, max_iter: int = 1000):
        self.scaler = StandardScaler()
        self.model  = LogisticRegression(
            max_iter=max_iter,
            class_weight="balanced",
            multi_class="multinomial",
            solver="lbfgs",
            C=1.0,
        )

    def fit(self, X: np.ndarray, y: np.ndarray):
        X_s = self.scaler.fit_transform(X)
        self.model.fit(X_s, y)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.model.predict(self.scaler.transform(X))

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self.model.predict_proba(self.scaler.transform(X))


class MLPLOB(nn.Module):
    """
    3-layer MLP for hand-crafted OFI features.
    Input: [B, n_feat]   (n_feat = n_levels + 2 from OFIDataset)
    Output: [B, 3]       logits
    """

    def __init__(self, n_feat: int, hidden: int = 64, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_feat, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


## 3c. CNN + LSTM (`models/cnn_lstm.py`)

Sequence-model baseline (Tsantekidis et al. 2017 style). Per-timestep Conv1d over the 10 LOB levels, then an LSTM over time.


In [14]:
"""
CNN + LSTM baseline.
Inspired by Tsantekidis et al. (2017), simplified and cleaned up.

Per-timestep 1D CNN over price levels → temporal LSTM.

Input:  [B, T, N, 4]
Output: [B, 3]  logits
"""

import torch
import torch.nn as nn


class CNNLSTM(nn.Module):
    """
    Args:
        n_levels   : number of LOB levels per side (default 10)
        d_cnn      : CNN output channels (feature extractor width)
        lstm_hidden: LSTM hidden size
        lstm_layers: LSTM depth
        dropout    : dropout probability
        n_classes  : number of output classes
    """

    def __init__(
        self,
        n_levels: int = 10,
        d_cnn: int = 64,
        lstm_hidden: int = 128,
        lstm_layers: int = 2,
        dropout: float = 0.2,
        n_classes: int = 3,
    ):
        super().__init__()
        in_channels = 4  # bid_p, bid_v, ask_p, ask_v per level

        # 1D CNN over price levels (applied identically at each time step)
        self.level_cnn = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=3, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(32),
            nn.Conv1d(32, d_cnn, kernel_size=3, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(d_cnn),
            nn.AdaptiveAvgPool1d(1),          # pool over N levels → [B*T, d_cnn, 1]
        )

        self.lstm = nn.LSTM(
            input_size=d_cnn,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(lstm_hidden, n_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, T, N, 4]"""
        B, T, N, C = x.shape

        # apply level CNN at each time step
        x = x.reshape(B * T, N, C)           # [B*T, N, 4]
        x = x.permute(0, 2, 1)               # [B*T, 4, N]  — channels first for Conv1d
        x = self.level_cnn(x)                # [B*T, d_cnn, 1]
        x = x.squeeze(-1)                    # [B*T, d_cnn]
        x = x.reshape(B, T, -1)              # [B, T, d_cnn]

        # LSTM over time
        _, (h, _) = self.lstm(x)             # h: [layers, B, hidden]
        out = self.dropout(h[-1])            # take last layer's hidden state
        return self.fc(out)                  # [B, 3]


## 3d. DeepLOB - Published SOTA Benchmark (`models/deeplob.py`)

Faithful reproduction of Zhang, Zohren & Roberts (2019) - IEEE TSP. Reference: arxiv.org/abs/1808.03668


In [15]:
"""
DeepLOB — Zhang et al. (2019)
"DeepLOB: Deep Convolutional Neural Networks for Limit Order Books"
IEEE Transactions on Signal Processing, 67(11), 3001-3012.

Canonical benchmark architecture reproduced faithfully.

Input:  [B, 1, T, 4*N]   (batch, 1 channel, time, 4*n_levels features)
Output: [B, 3]            logits (DOWN / STATIONARY / UP)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class InceptionModule(nn.Module):
    """
    Inception-style block operating along the time dimension.
    Three branches: 1×1 conv, 3×1 conv, 3×1 max-pool.
    """

    def __init__(self, in_channels: int, out_channels: int = 64):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(1, 1)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(out_channels),
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(3, 1), padding=(1, 0)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(out_channels),
        )
        # branch3: max-pool passes through channels unchanged
        self.branch3 = nn.MaxPool2d(kernel_size=(3, 1), stride=1, padding=(1, 0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        return torch.cat([b1, b2, b3], dim=1)   # [B, 64+64+C_in, T', F']


class DeepLOB(nn.Module):
    """
    DeepLOB network.

    Args:
        n_levels  : number of price levels per side (default 10)
        seq_len   : number of time steps (default 100)
        lstm_hidden: LSTM hidden units (default 64 per paper, 128 recommended)
        n_classes : output classes (default 3)
    """

    def __init__(self, n_levels: int = 10, seq_len: int = 100,
                 lstm_hidden: int = 64, n_classes: int = 3):
        super().__init__()
        in_features = n_levels * 4   # 40 for n_levels=10

        # ── CNN block ────────────────────────────────────────────────────────
        # Conv along feature dimension first (pairs bid/ask at each level)
        self.cnn = nn.Sequential(
            # halve feature dimension: 40 → 20
            nn.Conv2d(1, 32, kernel_size=(1, 2), stride=(1, 2)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(32),
            # reduce time
            nn.Conv2d(32, 32, kernel_size=(4, 1)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(32),
        )

        # ── Inception blocks ─────────────────────────────────────────────────
        # After CNN: channels=32, time reduced, features=n_levels*2
        # Inception1: 32 → 64+64+32=160
        self.inception1 = InceptionModule(32, out_channels=64)
        # Inception2: 160 → 64+64+160=288
        self.inception2 = InceptionModule(64 + 64 + 32, out_channels=64)

        # After 2 inception blocks channels = 64+64+(64+64+32) = 288
        inception_out_ch = 64 + 64 + (64 + 64 + 32)  # = 288

        # compute feature dimension after CNN
        feat_after_cnn = in_features // 2  # stride-2 conv: 40 → 20

        # ── LSTM ─────────────────────────────────────────────────────────────
        # Flatten spatial dimension: LSTM input = inception_out_ch * feat_after_cnn
        self.lstm_input_size = inception_out_ch * feat_after_cnn
        self.lstm = nn.LSTM(
            input_size=self.lstm_input_size,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
        )

        self.fc = nn.Linear(lstm_hidden, n_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="leaky_relu")
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [B, T, N, 4]  →  reshape to [B, 1, T, 4*N] for 2D CNN
        """
        B, T, N, C = x.shape
        x = x.reshape(B, T, N * C)           # [B, T, 40]
        x = x.unsqueeze(1)                    # [B, 1, T, 40]

        x = self.cnn(x)                       # [B, 32, T', 20]
        x = self.inception1(x)                # [B, 160, T', 20]
        x = self.inception2(x)                # [B, 288, T', 20]

        B2, C2, T2, F2 = x.shape
        x = x.permute(0, 2, 1, 3)            # [B, T', 288, 20]
        x = x.reshape(B2, T2, C2 * F2)       # [B, T', 288*20]

        _, (h, _) = self.lstm(x)              # h: [1, B, hidden]
        out = self.fc(h.squeeze(0))           # [B, 3]
        return out


## 3e. LOBTransformer - OUR CONTRIBUTION (`models/lob_transformer.py`)

Three-stage hierarchical 2D attention architecture:
- **Stage 1 - Level Encoder**: spatial attention within each snapshot
- **Stage 2 - Temporal Transformer**: attention across time
- **Stage 3 - Multi-task heads**: direction + volatility regime

807,046 parameters - 3.7x smaller than DeepLOB.


In [16]:
"""
LOBTransformer — Our Main Contribution
=======================================
Hierarchical 2D attention for Limit Order Book prediction.

Architecture:
  Stage 1 — Level Encoder  (spatial attention within each LOB snapshot)
    Linear projection per level → Multi-head self-attention over N levels
    → mean pool → projection to d_model

  Stage 2 — Temporal Transformer  (attention across T time steps)
    Sinusoidal positional encoding → TransformerEncoder

  Stage 3 — Multi-task heads
    direction_head   : Linear → 3 classes  (DOWN / STATIONARY / UP)
    volatility_head  : Linear → 3 classes  (low / med / high vol)
    Loss = L_direction + λ * L_volatility   (λ = 0.3)

Input:  [B, T, N, 4]
Output: (direction_logits [B,3],  vol_logits [B,3])
        During inference only direction_logits is used.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ── Positional encoding ───────────────────────────────────────────────────────

class SinusoidalPE(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float)
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# ── Level encoder ─────────────────────────────────────────────────────────────

class LevelEncoder(nn.Module):
    """
    Encodes each LOB snapshot [N, 4] into a d_model-dimensional vector.

    1. Project each level: [N, 4] → [N, d_level]
    2. Self-attention over N levels (spatial attention)
    3. Mean pool over N → [d_level]
    4. Linear → [d_model]
    """

    def __init__(self, n_levels: int = 10, d_level: int = 32,
                 d_model: int = 128, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.proj = nn.Linear(4, d_level)
        # level self-attention
        self.attn = nn.MultiheadAttention(
            embed_dim=d_level, num_heads=n_heads,
            dropout=dropout, batch_first=True
        )
        self.norm1 = nn.LayerNorm(d_level)
        self.ff    = nn.Sequential(
            nn.Linear(d_level, d_level * 2),
            nn.GELU(),
            nn.Linear(d_level * 2, d_level),
        )
        self.norm2 = nn.LayerNorm(d_level)
        self.out   = nn.Linear(d_level, d_model)
        self.drop  = nn.Dropout(dropout)

        self._attn_weights: torch.Tensor = None  # store for visualization

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B*T, N, 4]  →  [B*T, d_model]"""
        z = self.proj(x)                          # [B*T, N, d_level]
        attn_out, weights = self.attn(z, z, z)    # [B*T, N, d_level]
        self._attn_weights = weights.detach()
        z = self.norm1(z + self.drop(attn_out))
        z = self.norm2(z + self.drop(self.ff(z)))
        z = z.mean(dim=1)                         # [B*T, d_level]
        return self.out(z)                        # [B*T, d_model]


# ── LOBTransformer ────────────────────────────────────────────────────────────

class LOBTransformer(nn.Module):
    """
    Args:
        n_levels   : LOB levels per side (default 10)
        seq_len    : time steps per sample (default 100)
        d_level    : level encoder dimension
        d_model    : temporal transformer dimension
        n_heads    : attention heads (both level and temporal)
        n_layers   : temporal transformer depth
        dropout    : dropout rate
        n_classes  : direction classes (default 3)
        vol_lambda : weight for volatility auxiliary loss (default 0.3)
    """

    def __init__(
        self,
        n_levels: int = 10,
        seq_len:  int = 100,
        d_level:  int = 32,
        d_model:  int = 128,
        n_heads:  int = 8,
        n_layers: int = 4,
        dropout:  float = 0.1,
        n_classes: int = 3,
        vol_lambda: float = 0.3,
    ):
        super().__init__()
        self.vol_lambda = vol_lambda

        # Stage 1: level encoder
        self.level_encoder = LevelEncoder(
            n_levels=n_levels, d_level=d_level, d_model=d_model,
            n_heads=max(1, d_level // 8), dropout=dropout
        )

        # Stage 2: temporal transformer
        self.pos_enc = SinusoidalPE(d_model, max_len=seq_len + 10, dropout=dropout)
        import warnings
        warnings.filterwarnings("ignore", message="enable_nested_tensor")
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
            norm_first=True,               # pre-norm for stability
        )
        self.temporal_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers,
            norm=nn.LayerNorm(d_model),
        )

        # Stage 3: multi-task heads
        self.direction_head  = nn.Linear(d_model, n_classes)
        self.volatility_head = nn.Linear(d_model, 3)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor):
        """
        x: [B, T, N, 4]
        Returns: (direction_logits [B,3], vol_logits [B,3])
        """
        B, T, N, C = x.shape

        # Stage 1: encode each snapshot
        x_flat = x.reshape(B * T, N, C)           # [B*T, N, 4]
        enc    = self.level_encoder(x_flat)        # [B*T, d_model]
        enc    = enc.reshape(B, T, -1)             # [B, T, d_model]

        # Stage 2: temporal transformer
        enc = self.pos_enc(enc)                    # [B, T, d_model]
        enc = self.temporal_encoder(enc)           # [B, T, d_model]
        enc = enc.mean(dim=1)                      # [B, d_model]  (mean pooling)

        # Stage 3: heads
        dir_logits = self.direction_head(enc)      # [B, 3]
        vol_logits = self.volatility_head(enc)     # [B, 3]
        return dir_logits, vol_logits

    def get_level_attention(self) -> torch.Tensor:
        """Return last-forward level attention weights for visualization."""
        return self.level_encoder._attn_weights

    def compute_loss(
        self,
        dir_logits: torch.Tensor,
        vol_logits: torch.Tensor,
        dir_labels: torch.Tensor,
        vol_labels: torch.Tensor,
        class_weights: torch.Tensor = None,
    ) -> torch.Tensor:
        """Combined multi-task loss."""
        loss_dir = F.cross_entropy(dir_logits, dir_labels, weight=class_weights)
        loss_vol = F.cross_entropy(vol_logits, vol_labels)
        return loss_dir + self.vol_lambda * loss_vol


## 3f. BiGRU + Attention (`models/bigru_attention.py`)

Lightest sequence-model in the comparison. 182K parameters. Adapted from team prototype, integrated into our walk-forward + balanced-class pipeline.


In [17]:
"""
BiGRU + Temporal Attention.

Adapted from the team's original architecture (Cheekati / Pederson) but
integrated into our walk-forward, balanced-class, 2D-LOB pipeline.

Bidirectional GRU over a 100-tick LOB window, followed by multi-head
self-attention over time, then 3-class direction prediction.

Input:  [B, T, N, 4]
Output: [B, 3]  logits
"""

import torch
import torch.nn as nn


class BiGRUAttention(nn.Module):
    """
    Args:
        n_levels   : number of LOB levels per side (default 10)
        hidden     : GRU hidden size (per direction)
        n_layers   : GRU depth
        n_heads    : attention heads
        dropout    : dropout probability
        n_classes  : number of output classes
    """

    def __init__(
        self,
        n_levels: int = 10,
        hidden:   int = 64,
        n_layers: int = 2,
        n_heads:  int = 4,
        dropout:  float = 0.2,
        n_classes: int = 3,
    ):
        super().__init__()
        input_dim = n_levels * 4   # 40 features per timestep

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden * 2,        # bidirectional → ×2
            num_heads=n_heads,
            dropout=0.1,
            batch_first=True,
        )

        self.norm    = nn.LayerNorm(hidden * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden * 2, n_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, T, N, 4]"""
        B, T, N, C = x.shape

        # flatten level structure → standard sequence model input
        x = x.reshape(B, T, N * C)            # [B, T, 40]

        # bidirectional GRU
        gru_out, _ = self.gru(x)              # [B, T, 2*hidden]

        # temporal self-attention
        attn_out, _ = self.attn(gru_out, gru_out, gru_out)
        attn_out = self.norm(attn_out + gru_out)   # residual + norm

        # use last timestep representation
        last = self.dropout(attn_out[:, -1, :])
        return self.fc(last)                  # [B, 3]


---

## QA — Are All Reported Metrics Sensible?

Before any results are claimed, we run a full sanity-check on every reported metric. This section replicates that audit interactively.

### What we report (and why)

| Metric | Range | Why we use it |
|---|---|---|
| **Accuracy** | [0, 1] | Random baseline = 0.333 for 3-class. Easy to interpret. |
| **F1-Macro** | [0, 1] | Average of per-class F1 — robust to class imbalance. |
| **F1-Weighted** | [0, 1] | Per-class F1 weighted by support. |
| **Cohen's Kappa κ** | [-1, 1] | Agreement above chance. **κ = 0 means random.** Headline metric in LOB literature. |
| **MCC** | [-1, 1] | Correlation between predictions and labels. Like κ but symmetric in errors. |
| **Per-class F1** (DOWN / STAT / UP) | [0, 1] each | Reveals if the model is biased toward predicting one class. |

### What we deliberately DO NOT report

**MSE, RMSE, MAE, R-squared** — these are *regression* metrics. Our task is 3-class classification of return direction (not a continuous price target). Reporting MSE on classification logits would be a category error.

*(The team's earlier Dalton notebook reported MAE = 0.07 on a raw-mid-price regression task — but mid-price is heavily autocorrelated at 1Hz, so a trivial 'predict-last-value' baseline matches that score. Our quantile-balanced classification approach explicitly avoids this triviality.)*


### Live audit — run the same checks the project author ran


In [19]:
import json, os
import pandas as pd
from IPython.display import display, Markdown

# Get results directory from Config
cfg = Config()
RESULTS_DIR = cfg.results_dir

MODELS  = ['lr', 'mlp', 'cnn_lstm', 'deeplob', 'lob_transformer', 'bigru_attn']
HORIZONS = [10, 20, 50]

rows = []
for m in MODELS:
    for k in HORIZONS:
        p = os.path.join(RESULTS_DIR, f'{m}_BTC_k{k}.json')
        if not os.path.exists(p): continue
        r = json.load(open(p))['test']
        f1pc = r['f1_per_class']
        rows.append({
            'Model': m, 'Horizon': f'k={k}',
            'Accuracy':    r['accuracy'],
            'F1-Macro':    r['f1_macro'],
            'F1-Weighted': r.get('f1_weighted', float('nan')),
            'Cohen κ':     r['cohen_kappa'],
            'MCC':         r['mcc'],
            'F1-DOWN':     f1pc[0],
            'F1-STAT':     f1pc[1],
            'F1-UP':       f1pc[2],
        })
df = pd.DataFrame(rows)
display(df.style.hide(axis='index').format({
    'Accuracy': '{:.4f}', 'F1-Macro': '{:.4f}', 'F1-Weighted': '{:.4f}',
    'Cohen κ': '{:.4f}', 'MCC': '{:.4f}',
    'F1-DOWN': '{:.3f}', 'F1-STAT': '{:.3f}', 'F1-UP': '{:.3f}',
}).background_gradient(subset=['Cohen κ'], cmap='Greens'))


KeyError: "None of [Index(['Cohen κ'], dtype='object')] are in the [columns]"

In [ ]:
# Sanity-check rules — flag anything suspicious
issues = []
for _, r in df.iterrows():
    m, k = r['Model'], r['Horizon']
    if not (0.0 <= r['Accuracy'] <= 1.0):
        issues.append(f'{m} {k}: accuracy out of [0,1]')
    if not (-1.0 <= r['Cohen κ'] <= 1.0):
        issues.append(f'{m} {k}: kappa out of [-1,1]')
    if not (-1.0 <= r['MCC'] <= 1.0):
        issues.append(f'{m} {k}: MCC out of [-1,1]')
    if abs(r['Cohen κ'] - r['MCC']) > 0.05:
        issues.append(f'{m} {k}: kappa vs MCC differ by >0.05 (unusual on balanced data)')
    expected = (r['F1-DOWN'] + r['F1-STAT'] + r['F1-UP']) / 3
    if abs(r['F1-Macro'] - expected) > 1e-3:
        issues.append(f'{m} {k}: F1-macro != mean(per-class F1)')
    if r['Accuracy'] < 0.30:
        issues.append(f'{m} {k}: accuracy below random 0.333')
    if r['Cohen κ'] < 0.10 and m != 'lr':
        issues.append(f'{m} {k}: kappa suspiciously low for non-baseline')

if issues:
    display(Markdown('### ⚠️ Issues flagged'))
    for i in issues:
        print('  -', i)
else:
    display(Markdown('### ✅ All metrics within expected bounds'))
    display(Markdown('Every saved metric falls in its valid range; '
                     'F1-Macro reconciles with per-class F1; '
                     'Kappa and MCC agree (as expected for balanced classes); '
                     'all models beat random.'))


### Key observations from the audit

| Finding | Interpretation |
|---|---|
| **LR is consistent across horizons** (κ ≈ 0.220) | Linear model has no temporal modeling — horizon doesn't matter. Healthy sign. |
| **All 4 deep models cluster tightly** at κ 0.37–0.38 | Architecture differences are within noise floor. Protocol matters more than architecture for this signal. |
| **Per-class F1 always orders: STAT > UP > DOWN** | Slight negative-direction bias in the Feb 21–28 BTC test window. Reflects market regime, not model bug. |
| **DeepLOB k=20 dipped to κ = 0.366** | 1 pp lower than k=10 / k=50. Likely random-init variance — would shrink with multi-seed averaging. |
| **BiGRU has highest MCC (0.380)** at k=10 | Slightly better balance of false positives/negatives than LOBTransformer. |
| **LOBTransformer beats DeepLOB on accuracy** at k=10 (0.587 vs 0.585) | At 3.7× fewer parameters — validates the hierarchical-attention design choice. |

### What about the backtest metrics (Sharpe, MaxDD, P&L)?

Those are interpreted in Section 8. Note that Sharpe values of 4–5 are inflated by:

1. **Period (not annualised) Sharpe** computed over a 7-day test window
2. **HFT central limit theorem effect** — at ~50 trades/day with consistent edge, daily P&L is the sum of many small bets, so daily variance shrinks and Sharpe inflates
3. **Fixed-position-size assumption** — caps per-trade loss, suppresses drawdown

We treat Sharpe values in this notebook as **upper bounds** on real-world performance. Realistic deployment would face latency, queue priority, market impact, and inventory management — none of which our backtest models. This is documented as a limitation in the Future Work section.


---


## 4. Training Loop (`train.py`)

AdamW + CosineAnnealingLR, class-weighted CrossEntropy, multi-task aux loss, early stopping on validation Cohen's Kappa, best-checkpoint save.


In [ ]:
"""
Training pipeline with walk-forward cross-validation.

Usage (via run.py):
    python run.py train --model lob_transformer --symbol BTC --horizon 10
    python run.py train --model deeplob         --symbol BTC --horizon 10 --debug
"""

import os
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from config import Config
from data.dataset import (LOBDataset, OFIDataset, FeatureNormalizer,
                           walk_forward_split)
from models.baseline import MLPLOB, LogisticRegressionLOB
from models.deeplob import DeepLOB
from models.cnn_lstm import CNNLSTM
from models.lob_transformer import LOBTransformer
from models.bigru_attention import BiGRUAttention
from utils import compute_class_weights, compute_metrics, print_metrics, save_results


# -- model factory --------------------------------------------------------------

def build_model(config: Config) -> nn.Module:
    name = config.model_name
    if name == "mlp":
        return MLPLOB(n_feat=config.n_levels + 2).to(config.device)
    if name == "cnn_lstm":
        return CNNLSTM(n_levels=config.n_levels,
                       lstm_hidden=config.lstm_hidden).to(config.device)
    if name == "deeplob":
        return DeepLOB(n_levels=config.n_levels, seq_len=config.seq_len,
                       lstm_hidden=config.lstm_hidden).to(config.device)
    if name == "lob_transformer":
        return LOBTransformer(
            n_levels=config.n_levels, seq_len=config.seq_len,
            d_level=config.d_level, d_model=config.d_model,
            n_heads=config.n_heads, n_layers=config.n_layers,
            dropout=config.dropout,
        ).to(config.device)
    if name == "bigru_attn":
        return BiGRUAttention(
            n_levels=config.n_levels,
            hidden=64, n_layers=2, n_heads=4,
            dropout=config.dropout,
        ).to(config.device)
    raise ValueError(f"Unknown model: {name}")


# -- dataset factory ------------------------------------------------------------

def build_datasets(config: Config, symbol: str,
                   train_idx: np.ndarray,
                   val_idx:   np.ndarray,
                   test_idx:  np.ndarray):
    """
    Build train/val/test DataLoaders.
    Normalizer is fit on training rows only.
    """
    parquet_dir = os.path.join(config.processed_dir, symbol)
    use_ofi     = config.model_name in ("lr", "mlp")

    # fit normalizer
    normalizer = FeatureNormalizer()
    if use_ofi:
        tmp = OFIDataset(parquet_dir, config.label_col, config.n_levels,
                         normalizer=None, indices=train_idx)
    else:
        tmp = LOBDataset(parquet_dir, config.seq_len, config.label_col,
                         config.n_levels, normalizer=None, indices=train_idx)
    # fit on features at training window positions
    normalizer.fit(tmp.X[train_idx])

    def _make(idx, shuffle):
        if use_ofi:
            ds = OFIDataset(parquet_dir, config.label_col, config.n_levels,
                            normalizer=normalizer, indices=idx)
        else:
            ds = LOBDataset(parquet_dir, config.seq_len, config.label_col,
                            config.n_levels, normalizer=normalizer, indices=idx)
        return DataLoader(ds, batch_size=config.batch_size, shuffle=shuffle,
                          num_workers=config.num_workers,
                          pin_memory=(config.device == "cuda"))

    return _make(train_idx, True), _make(val_idx, False), _make(test_idx, False)


# -- one epoch -----------------------------------------------------------------

def _forward(model, X, y_dir, y_vol, class_weights, config):
    """Unified forward + loss for all model types."""
    if config.model_name == "lob_transformer":
        dir_logits, vol_logits = model(X)
        loss = model.compute_loss(dir_logits, vol_logits,
                                  y_dir, y_vol, class_weights)
        preds = dir_logits.argmax(dim=1)
    else:
        logits = model(X)
        loss   = nn.functional.cross_entropy(logits, y_dir, weight=class_weights)
        preds  = logits.argmax(dim=1)
    return loss, preds


def train_epoch(model, loader, optimizer, class_weights, config):
    model.train()
    total_loss, all_pred, all_true = 0.0, [], []

    for batch in loader:
        X, y_dir, y_vol = batch
        X, y_dir, y_vol = (X.to(config.device), y_dir.to(config.device),
                           y_vol.to(config.device))

        optimizer.zero_grad()
        loss, preds = _forward(model, X, y_dir, y_vol, class_weights, config)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(y_dir)
        all_pred.extend(preds.cpu().numpy())
        all_true.extend(y_dir.cpu().numpy())

    m = compute_metrics(np.array(all_true), np.array(all_pred))
    m["loss"] = total_loss / max(len(all_true), 1)
    return m


@torch.no_grad()
def eval_epoch(model, loader, class_weights, config):
    model.eval()
    total_loss, all_pred, all_true = 0.0, [], []

    for batch in loader:
        X, y_dir, y_vol = batch
        X, y_dir, y_vol = (X.to(config.device), y_dir.to(config.device),
                           y_vol.to(config.device))

        loss, preds = _forward(model, X, y_dir, y_vol, class_weights, config)

        total_loss += loss.item() * len(y_dir)
        all_pred.extend(preds.cpu().numpy())
        all_true.extend(y_dir.cpu().numpy())

    m = compute_metrics(np.array(all_true), np.array(all_pred))
    m["loss"] = total_loss / max(len(all_true), 1)
    return m


# -- main training function -----------------------------------------------------

def train(config: Config, symbol: str = "BTC"):
    torch.manual_seed(config.seed)
    np.random.seed(config.seed)

    parquet_dir = os.path.join(config.processed_dir, symbol)
    if not os.path.isdir(parquet_dir):
        raise RuntimeError(
            f"Preprocessed data not found at {parquet_dir}. "
            "Run: python run.py preprocess --symbol " + symbol
        )

    train_idx, val_idx, test_idx = walk_forward_split(parquet_dir, config.seq_len)

    print(f"\n{'='*60}")
    print(f"Train  model={config.model_name}  symbol={symbol}  "
          f"horizon=k{config.horizon_key}  device={config.device}")
    print(f"  windows -> train={len(train_idx):,}  val={len(val_idx):,}  "
          f"test={len(test_idx):,}")
    print(f"{'='*60}\n")

    # -- sklearn logistic regression ---------------------------------------
    if config.model_name == "lr":
        return _train_lr(config, symbol, train_idx, val_idx, test_idx)

    train_loader, val_loader, test_loader = build_datasets(
        config, symbol, train_idx, val_idx, test_idx
    )

    # class weights from training labels
    all_y = np.concatenate([b[1].numpy() for b in train_loader])
    cw    = compute_class_weights(all_y, device=config.device)
    print(f"Class weights: DOWN={cw[0]:.3f}  STAT={cw[1]:.3f}  UP={cw[2]:.3f}\n")

    model    = build_model(config)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {config.model_name}  trainable params={n_params:,}\n")

    optimizer = AdamW(model.parameters(), lr=config.lr,
                      weight_decay=config.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=config.epochs, eta_min=1e-6)

    best_kappa  = -1.0
    patience_ct = 0
    ckpt_path   = os.path.join(
        config.save_dir,
        f"{config.model_name}_{symbol}_k{config.horizon_key}_best.pt"
    )
    history = []

    for epoch in range(1, config.epochs + 1):
        t0 = time.time()
        tr = train_epoch(model, train_loader, optimizer, cw, config)
        vl = eval_epoch(model,  val_loader,   cw, config)
        scheduler.step()

        history.append({"epoch": epoch, "train": tr, "val": vl})
        print(f"Epoch {epoch:3d}/{config.epochs}  ({time.time()-t0:.1f}s)")
        print_metrics(tr, prefix="Train")
        print_metrics(vl, prefix="Val  ")
        print()

        if vl["cohen_kappa"] > best_kappa:
            best_kappa  = vl["cohen_kappa"]
            patience_ct = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_ct += 1
            if patience_ct >= config.patience:
                print(f"Early stopping at epoch {epoch}")
                break

    # -- test evaluation ---------------------------------------------------
    model.load_state_dict(torch.load(ckpt_path, map_location=config.device,
                                     weights_only=True))
    test_m = eval_epoch(model, test_loader, cw, config)
    print("\nTest results:")
    print_metrics(test_m, prefix="Test ")

    results = {"model": config.model_name, "symbol": symbol,
               "horizon": config.horizon_key, "test": test_m, "history": history}
    save_results(results, os.path.join(
        config.results_dir,
        f"{config.model_name}_{symbol}_k{config.horizon_key}.json"
    ))
    return results


# -- logistic regression --------------------------------------------------------

def _train_lr(config: Config, symbol: str,
              train_idx: np.ndarray, val_idx: np.ndarray, test_idx: np.ndarray):
    parquet_dir = os.path.join(config.processed_dir, symbol)

    normalizer = FeatureNormalizer()
    raw = OFIDataset(parquet_dir, config.label_col, config.n_levels,
                     normalizer=None)
    normalizer.fit(raw.X[train_idx])

    def _get_xy(idx):
        ds = OFIDataset(parquet_dir, config.label_col, config.n_levels,
                        normalizer=normalizer, indices=idx)
        return ds.X[idx], ds.y_dir[idx]

    X_tr, y_tr = _get_xy(train_idx)
    X_vl, y_vl = _get_xy(val_idx)
    X_te, y_te = _get_xy(test_idx)

    lr = LogisticRegressionLOB()
    print("Fitting Logistic Regression ...")
    lr.fit(X_tr, y_tr)

    for X, y, name in [(X_vl, y_vl, "Val"), (X_te, y_te, "Test")]:
        m = compute_metrics(y, lr.predict(X))
        print_metrics(m, prefix=name)

    test_m = compute_metrics(y_te, lr.predict(X_te))
    results = {"model": "lr", "symbol": symbol, "horizon": config.horizon_key,
               "test": test_m}
    save_results(results, os.path.join(
        config.results_dir, f"lr_{symbol}_k{config.horizon_key}.json"
    ))
    return results


## 5. Evaluation (`evaluate.py`)

Test-set metrics, per-regime breakdown, confusion matrices, and attention heatmaps.


In [ ]:
"""
Evaluation: model comparison, per-regime analysis, attention heatmaps.

Usage:
    python run.py evaluate --model lob_transformer --symbol BTC --horizon 10
    python run.py evaluate --compare-all --symbol BTC --horizon 10
"""

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

from config import Config
from data.dataset import LOBDataset, FeatureNormalizer, walk_forward_split
from train import build_model, build_datasets
from utils import compute_metrics, print_metrics, load_results, CLASS_NAMES


# -- inference -----------------------------------------------------------------

@torch.no_grad()
def run_inference(config: Config, symbol: str):
    """Load best checkpoint, return (y_true, y_pred, attn_weights|None)."""
    parquet_dir = os.path.join(config.processed_dir, symbol)
    train_idx, val_idx, test_idx = walk_forward_split(parquet_dir, config.seq_len)
    _, _, test_loader = build_datasets(config, symbol, train_idx, val_idx, test_idx)

    ckpt = os.path.join(config.save_dir,
                        f"{config.model_name}_{symbol}_k{config.horizon_key}_best.pt")
    if not os.path.isfile(ckpt):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt}. Train first.")

    model = build_model(config)
    model.load_state_dict(torch.load(ckpt, map_location=config.device,
                                     weights_only=True))
    model.eval()

    all_true, all_pred, all_attn = [], [], []
    for batch in test_loader:
        X, y_dir, _ = batch
        X = X.to(config.device)
        if config.model_name == "lob_transformer":
            logits, _ = model(X)
            attn = model.get_level_attention()
            if attn is not None:
                all_attn.append(attn.cpu().numpy())
        else:
            logits = model(X)
        all_pred.extend(logits.argmax(dim=1).cpu().numpy())
        all_true.extend(y_dir.numpy())

    attn_arr = np.concatenate(all_attn, axis=0) if all_attn else None
    return np.array(all_true), np.array(all_pred), attn_arr


# -- per-regime breakdown -------------------------------------------------------

def regime_breakdown(config: Config, symbol: str,
                     y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    parquet_dir = os.path.join(config.processed_dir, symbol)
    files = sorted([os.path.join(parquet_dir, f)
                    for f in os.listdir(parquet_dir) if f.endswith(".parquet")])
    df = pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)

    _, _, test_idx = walk_forward_split(parquet_dir, config.seq_len)
    end_idx = (test_idx + config.seq_len - 1).clip(max=len(df) - 1)
    n       = min(len(end_idx), len(y_true))
    vol_lbl = df["vol_label"].to_numpy()[end_idx[:n]]

    results = {}
    for regime, name in enumerate(["Low Vol", "Med Vol", "High Vol"]):
        mask = vol_lbl == regime
        if mask.sum() == 0:
            continue
        m = compute_metrics(y_true[:n][mask], y_pred[:n][mask])
        results[name] = m
        print(f"  {name:10s}: Acc={m['accuracy']:.4f}  "
              f"F1={m['f1_macro']:.4f}  Kappa={m['cohen_kappa']:.4f}")
    return results


# -- plots ----------------------------------------------------------------------

def plot_confusion_matrix(y_true, y_pred, title: str, save_path: str):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2], normalize="true")
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(title); ax.set_ylabel("True"); ax.set_xlabel("Predicted")
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()
    print(f"Saved {save_path}")


def plot_attention_heatmap(attn_weights: np.ndarray, save_path: str,
                            n_levels: int = 10):
    avg = attn_weights[:500].mean(axis=0)         # [N, N]
    fig, ax = plt.subplots(figsize=(6, 5))
    lbls = [f"L{i}" for i in range(n_levels)]
    sns.heatmap(avg, annot=True, fmt=".2f", cmap="YlOrRd",
                xticklabels=lbls, yticklabels=lbls, ax=ax)
    ax.set_title("Level Attention Weights\n(avg over test samples, all heads)")
    ax.set_ylabel("Query level"); ax.set_xlabel("Key level")
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()
    print(f"Saved {save_path}")


def plot_training_curves(history: list, save_path: str, model_name: str):
    epochs    = [h["epoch"] for h in history]
    tr_acc    = [h["train"]["accuracy"] for h in history]
    vl_acc    = [h["val"]["accuracy"]   for h in history]
    tr_kappa  = [h["train"]["cohen_kappa"] for h in history]
    vl_kappa  = [h["val"]["cohen_kappa"]   for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, tr_acc,   label="Train Acc")
    ax1.plot(epochs, vl_acc,   label="Val Acc", linestyle="--")
    ax1.set_title(f"{model_name} - Accuracy"); ax1.legend()
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")

    ax2.plot(epochs, tr_kappa,  label="Train Kappa")
    ax2.plot(epochs, vl_kappa,  label="Val Kappa", linestyle="--")
    ax2.set_title(f"{model_name} - Cohen's Kappa"); ax2.legend()
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Kappa")

    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()
    print(f"Saved {save_path}")


def plot_model_comparison(results_dir: str, symbol: str, horizon: int, save_path: str):
    model_order = ["lr", "mlp", "cnn_lstm", "deeplob", "lob_transformer", "bigru_attn"]
    rows = []
    for m in model_order:
        fp = os.path.join(results_dir, f"{m}_{symbol}_k{horizon}.json")
        if os.path.isfile(fp):
            r = load_results(fp)
            rows.append({"Model": m, **{k: r["test"][k]
                          for k in ("accuracy", "f1_macro", "cohen_kappa", "mcc")}})

    if not rows:
        print("No result files found."); return

    df      = pd.DataFrame(rows)
    metrics = ["accuracy", "f1_macro", "cohen_kappa", "mcc"]
    labels  = ["Accuracy", "F1-Macro", "Cohen Kappa", "MCC"]
    colors  = plt.cm.tab10(np.linspace(0, 1, len(df)))

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for ax, col, lbl in zip(axes, metrics, labels):
        bars = ax.bar(df["Model"], df[col], color=colors)
        ax.set_title(lbl); ax.set_ylim(0, 1)
        ax.set_xticklabels(df["Model"], rotation=35, ha="right")
        for bar, val in zip(bars, df[col]):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.01,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=8)

    fig.suptitle(f"Model Comparison - {symbol}, horizon k={horizon}", fontsize=13)
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()
    print(f"Saved {save_path}")


# -- main evaluate function -----------------------------------------------------

def evaluate(config: Config, symbol: str = "BTC", compare_all: bool = False):
    if compare_all:
        print(f"\n{'='*60}")
        print(f"Model Comparison - {symbol}  horizon=k{config.horizon_key}")
        print(f"{'='*60}")
        model_list   = ["lr", "mlp", "cnn_lstm", "deeplob", "lob_transformer", "bigru_attn"]
        all_results  = {}
        for m in model_list:
            fp = os.path.join(config.results_dir,
                              f"{m}_{symbol}_k{config.horizon_key}.json")
            if os.path.isfile(fp):
                r = load_results(fp)
                print(f"\n[{m}]")
                print_metrics(r["test"])
                all_results[m] = r["test"]
            else:
                print(f"[{m}] - not trained yet")

        if all_results:
            print(f"\n{'-'*70}")
            print(f"{'Model':<20} {'Accuracy':>10} {'F1-Macro':>10} "
                  f"{'Kappa':>10} {'MCC':>10}")
            print(f"{'-'*70}")
            for m, mt in all_results.items():
                print(f"{m:<20} {mt['accuracy']:>10.4f} {mt['f1_macro']:>10.4f} "
                      f"{mt['cohen_kappa']:>10.4f} {mt['mcc']:>10.4f}")

        plot_model_comparison(
            config.results_dir, symbol, config.horizon_key,
            os.path.join(config.results_dir,
                         f"comparison_{symbol}_k{config.horizon_key}.png")
        )
        return all_results

    # single-model evaluation
    print(f"\nEvaluating {config.model_name} on {symbol} k{config.horizon_key} ...")
    y_true, y_pred, attn = run_inference(config, symbol)
    metrics = compute_metrics(y_true, y_pred)
    print_metrics(metrics, prefix=config.model_name)

    print("\nPer-regime breakdown:")
    regime_breakdown(config, symbol, y_true, y_pred)

    pfx = f"{config.model_name}_{symbol}_k{config.horizon_key}"
    plot_confusion_matrix(y_true, y_pred,
                          f"{config.model_name} - {symbol} k={config.horizon_key}",
                          os.path.join(config.results_dir, f"cm_{pfx}.png"))

    if attn is not None:
        plot_attention_heatmap(
            attn, n_levels=config.n_levels,
            save_path=os.path.join(config.results_dir, f"attn_{pfx}.png")
        )

    # training curves (if history available)
    fp = os.path.join(config.results_dir, f"{config.model_name}_{symbol}_k{config.horizon_key}.json")
    if os.path.isfile(fp):
        r = load_results(fp)
        if r.get("history"):
            plot_training_curves(
                r["history"], model_name=config.model_name,
                save_path=os.path.join(config.results_dir, f"curves_{pfx}.png")
            )


## 6. Backtest (`backtest.py`)

Multi-fee scenario backtest: zero_friction / vip_maker / retail_taker. Includes confidence threshold, max-trades-per-day cap, short-borrow modeling, and per-trade economic diagnostic.


In [ ]:
"""
Backtesting: simulate directional trading strategy from model predictions.

Three fee scenarios are simulated for every model and plotted side-by-side:

  no_fees        : 0.000%  per side  (theoretical signal quality)
  market_maker   : -0.0025% per side (Binance VIP-0 maker rebate)
  taker          : +0.05%  per side  (Binance standard taker fee)

Per-trade economics:
  net_return = direction * (exit - entry)/entry  -  2 * fee_per_side

Strategy:
  UP   (2) -> LONG  at mid_price; exit after k seconds at mid_price
  DOWN (0) -> SHORT at mid_price; exit after k seconds at mid_price
  STAT (1) -> no trade
  Optional: confidence threshold (only trade when max softmax > tau)
  One position at a time (no overlapping trades).

Entry/exit at mid_price models a limit-order strategy.
Bid/ask spread is already captured in the normalised features the model sees.

Usage:
    python run.py backtest --model lob_transformer --symbol BTC --horizon 10
    python run.py backtest --compare-all --symbol BTC --horizon 10
    python run.py backtest --compare-all --symbol BTC --horizon 10 --confidence 0.55
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from config import Config
from data.dataset import LOBDataset, FeatureNormalizer, walk_forward_split
from train import build_model, build_datasets
from utils import load_results


# Three economic regimes — these test the SAME directional alpha strategy at
# different fee tiers/execution styles. NOT actual market-making (which would
# quote both sides at microsecond latency).
#
#  zero_friction   : pure-signal baseline (no fees, no slippage)
#  vip_maker       : VIP-3 maker tier (-0.0025% rebate), limit-order fills, no slippage
#  retail_taker    : retail Binance taker (+0.05% fee) + half-spread slippage
FEE_SCENARIOS = {
    "zero_friction": {"fee": 0.0,        "slippage": 0.0},
    "vip_maker":     {"fee": -0.000025,  "slippage": 0.0},
    "retail_taker":  {"fee": 0.0005,     "slippage": 0.000025},
}

SCENARIO_COLORS = {
    "zero_friction": "tab:green",
    "vip_maker":     "tab:blue",
    "retail_taker":  "tab:red",
}


# -- inference with price metadata + confidence scores --------------------------

@torch.no_grad()
def get_predictions_with_prices(config: Config, symbol: str) -> pd.DataFrame:
    """
    Return test-set DataFrame with columns:
      timestamp, mid_price, best_bid, best_ask,
      y_true, y_pred, prob_down, prob_stat, prob_up, confidence
    """
    parquet_dir = os.path.join(config.processed_dir, symbol)
    train_idx, val_idx, test_idx = walk_forward_split(parquet_dir, config.seq_len)
    _, _, test_loader = build_datasets(config, symbol, train_idx, val_idx, test_idx)

    ckpt = os.path.join(config.save_dir,
                        f"{config.model_name}_{symbol}_k{config.horizon_key}_best.pt")
    if not os.path.isfile(ckpt):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt}")

    model = build_model(config)
    model.load_state_dict(torch.load(ckpt, map_location=config.device,
                                     weights_only=True))
    model.eval()

    all_pred, all_true, all_probs = [], [], []
    for batch in test_loader:
        X, y_dir, _ = batch
        X = X.to(config.device)
        if config.model_name == "lob_transformer":
            out = model(X)
            logits = out[0] if isinstance(out, tuple) else out
        else:
            logits = model(X)

        probs = F.softmax(logits, dim=1).cpu().numpy()   # [B, 3]
        all_probs.extend(probs.tolist())
        all_pred.extend(probs.argmax(axis=1).tolist())
        all_true.extend(y_dir.numpy().tolist())

    files   = sorted([os.path.join(parquet_dir, f)
                      for f in os.listdir(parquet_dir) if f.endswith(".parquet")])
    df_full = pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)

    end_idx = (test_idx + config.seq_len - 1).clip(max=len(df_full) - 1)
    n       = min(len(end_idx), len(all_pred))

    meta = df_full.iloc[end_idx[:n]][
        ["timestamp", "mid_price", "best_bid", "best_ask"]
    ].copy().reset_index(drop=True)

    probs_arr          = np.array(all_probs[:n])
    meta["y_true"]     = np.array(all_true[:n])
    meta["y_pred"]     = np.array(all_pred[:n])
    meta["prob_down"]  = probs_arr[:, 0]
    meta["prob_stat"]  = probs_arr[:, 1]
    meta["prob_up"]    = probs_arr[:, 2]
    meta["confidence"] = probs_arr.max(axis=1)
    return meta


# -- simulation ----------------------------------------------------------------

def simulate_strategy(df: pd.DataFrame, horizon: int,
                      tc: float = 0.0005,
                      confidence_threshold: float = 0.0,
                      max_trades_per_day: int = None) -> pd.DataFrame:
    """
    Simulate the directional strategy for a given per-side fee `tc`.

    Args:
        df                   : predictions DataFrame (output of get_predictions_with_prices)
        horizon              : seconds held per trade
        tc                   : per-side cost (fee + slippage combined)
        confidence_threshold : minimum max-softmax to consider a signal
        max_trades_per_day   : if set, only the top-N highest-confidence signals
                               per UTC day are eligible (operationally realistic)

    Returns a per-trade DataFrame with columns:
      entry_time, exit_time, direction, entry_price, exit_price,
      gross_return, return, cumulative_return
    """
    df = df.copy()

    # ── trade-frequency cap: keep only the top-N most confident
    # directional signals each day ─────────────────────────────────
    if max_trades_per_day is not None and max_trades_per_day > 0:
        df["date"]      = pd.to_datetime(df["timestamp"]).dt.date
        is_directional  = df["y_pred"].isin([0, 2]) & (df["confidence"] >= confidence_threshold)
        eligible_idx    = []
        for _, grp in df[is_directional].groupby("date"):
            top = grp.nlargest(max_trades_per_day, "confidence").index
            eligible_idx.extend(top.tolist())
        eligible = df.index.isin(eligible_idx)
        df.loc[~eligible, "y_pred"]     = 1     # neutralise non-eligible signals
        df.loc[~eligible, "confidence"] = 0.0

    trades      = []
    in_trade    = False
    entry_price = 0.0
    direction   = 0
    entry_idx   = 0

    mid    = df["mid_price"].to_numpy()
    sigs   = df["y_pred"].to_numpy()
    confs  = df["confidence"].to_numpy()
    times  = df["timestamp"].to_numpy()

    # Binance Margin BTC borrow rate: ~0.05% per day = ~5.8e-9 per second.
    # Only applied to SHORT positions (borrowing BTC to sell).
    SHORT_BORROW_PER_SEC = 0.0005 / 86400.0

    for i in range(len(df)):
        if in_trade and (i - entry_idx) >= horizon:
            exit_price = mid[i]
            raw_ret    = direction * (exit_price - entry_price) / entry_price
            # Costs: fee+slippage round-trip + borrow cost for shorts only
            cost = 2 * tc
            if direction == -1:
                cost += SHORT_BORROW_PER_SEC * horizon
            net_ret = raw_ret - cost
            trades.append({
                "entry_time":   times[entry_idx],
                "exit_time":    times[i],
                "direction":    "LONG" if direction == 1 else "SHORT",
                "entry_price":  entry_price,
                "exit_price":   exit_price,
                "gross_return": raw_ret,
                "return":       net_ret,
            })
            in_trade = False

        if not in_trade and confs[i] >= confidence_threshold:
            if sigs[i] == 2:
                in_trade, entry_price, direction, entry_idx = True, mid[i], 1, i
            elif sigs[i] == 0:
                in_trade, entry_price, direction, entry_idx = True, mid[i], -1, i

    if not trades:
        return pd.DataFrame(columns=["entry_time", "exit_time", "direction",
                                     "entry_price", "exit_price",
                                     "gross_return", "return"])

    tdf = pd.DataFrame(trades)
    # ADDITIVE (fixed-position) — each trade = fixed % of starting capital,
    # not compounded. Avoids the unrealistic geometric blow-up from 40K trades.
    tdf["cumulative_return"] = tdf["return"].cumsum()
    return tdf


# -- metrics -------------------------------------------------------------------

def backtest_metrics(tdf: pd.DataFrame) -> dict:
    """
    HFT-appropriate metrics:
      - total_return: sum of per-trade % returns (fixed-position assumption)
      - sharpe: annualised from DAILY rolled-up returns, scaled by sqrt(252)
                (independent of trade count — robust to HFT compounding artifact)
      - max_drawdown: peak-to-trough on the additive equity curve
    """
    if len(tdf) == 0:
        return {"n_trades": 0, "win_rate": 0.0, "total_return": 0.0,
                "sharpe": 0.0, "max_drawdown": 0.0, "mean_return": 0.0,
                "avg_gross_return": 0.0, "trades_per_day": 0.0}

    r     = tdf["return"].to_numpy()
    gross = tdf["gross_return"].to_numpy() if "gross_return" in tdf.columns else r

    # additive equity curve (no compounding)
    cum   = r.cumsum()
    peak  = np.maximum.accumulate(cum)
    dd    = cum - peak                                     # already in % terms

    # PERIOD Sharpe — no √252 annualisation; this is the test period itself.
    # Computed on daily-rolled-up returns: Sharpe = mean(daily) / std(daily)
    sharpe = 0.0
    trades_per_day = 0.0
    if "exit_time" in tdf.columns and len(tdf) > 1:
        ex = pd.to_datetime(tdf["exit_time"])
        days = ex.dt.floor("D")
        daily = tdf.groupby(days)["return"].sum()
        if daily.std() > 0:
            sharpe = float(daily.mean() / daily.std())   # period Sharpe
        n_days = max((ex.max() - ex.min()).total_seconds() / 86400.0, 1.0)
        trades_per_day = float(len(tdf) / n_days)

    return {
        "n_trades":         int(len(tdf)),
        "win_rate":         float((r > 0).mean()),
        "total_return":     float(cum[-1]),               # SUM of % returns
        "sharpe":           sharpe,                       # ANNUALISED
        "max_drawdown":     float(dd.min()),
        "mean_return":      float(r.mean()),
        "avg_gross_return": float(gross.mean()),
        "trades_per_day":   trades_per_day,
    }


# -- diagnostic: actual per-trade economics ------------------------------------

def diagnose_signal_economics(df: pd.DataFrame, horizon: int,
                               confidence_threshold: float = 0.55):
    """
    Print the per-trade economic reality: what is the model actually predicting,
    what does each trade win/lose, and how does that compare to costs.

    This is the cleanest explanation of why aggregate returns look the way
    they do.
    """
    mid   = df["mid_price"].to_numpy()
    sigs  = df["y_pred"].to_numpy()
    confs = df["confidence"].to_numpy()

    # gross return per trade, IF we entered at every signal (no overlap logic)
    ret = np.zeros(len(df), dtype=np.float64)
    valid = np.zeros(len(df), dtype=bool)
    for i in range(len(df) - horizon):
        if confs[i] < confidence_threshold:
            continue
        if sigs[i] not in (0, 2):
            continue
        d = 1 if sigs[i] == 2 else -1
        ret[i] = d * (mid[i + horizon] - mid[i]) / mid[i]
        valid[i] = True

    g = ret[valid] * 100   # convert to percent

    if len(g) == 0:
        print("\nNo qualifying signals at this confidence threshold.")
        return

    n_signals = len(g)
    test_n    = len(df)

    print(f"\n{'='*78}")
    print(f"Per-trade economics diagnostic  (k={horizon}s, conf>={confidence_threshold})")
    print(f"{'='*78}")
    print(f"Test set size              : {test_n:,} snapshots")
    print(f"Qualifying signals         : {n_signals:,} "
          f"({n_signals/test_n*100:.1f}% of snapshots)")
    print()
    print(f"GROSS per-trade return distribution (% — before any cost):")
    print(f"  mean                     : {g.mean():+.4f}%   (this is your edge)")
    print(f"  median                   : {np.median(g):+.4f}%")
    print(f"  std                      : {g.std():+.4f}%")
    print(f"  10th / 90th percentile   : {np.percentile(g, 10):+.4f}%  /  "
          f"{np.percentile(g, 90):+.4f}%")
    print(f"  win rate (gross > 0)     : {(g > 0).mean()*100:.1f}%")
    print()
    print(f"Cost vs. edge comparison (round-trip):")
    print(f"  Average gross edge       : {g.mean():+.4f}%")
    for scen, c in FEE_SCENARIOS.items():
        rt = (c["fee"] + c["slippage"]) * 2 * 100
        net = g.mean() - rt
        verdict = "PROFITABLE" if net > 0 else "UNPROFITABLE"
        print(f"  {scen:<14} round-trip = {rt:+.4f}%  -> "
              f"net edge = {net:+.4f}%  [{verdict}]")
    print()
    print(f"Of all signals, % where |gross| > round-trip cost:")
    for scen, c in FEE_SCENARIOS.items():
        rt = (c["fee"] + c["slippage"]) * 2 * 100
        pct = (np.abs(g) > rt).mean() * 100
        # of the signals that DO exceed cost, are they correctly directional?
        big = np.abs(g) > rt
        if big.sum() > 0:
            win_among_big = (g[big] > 0).mean() * 100
        else:
            win_among_big = 0.0
        print(f"  {scen:<14} cost={rt:+.4f}%  -> "
              f"{pct:5.1f}% of signals are large enough  "
              f"(of those, {win_among_big:.1f}% would profit)")
    print()
    print(f"Why the aggregate returns look like they do:")
    daily_trades = n_signals / 7   # rough — test set ≈ 7 days
    for scen, c in FEE_SCENARIOS.items():
        rt = (c["fee"] + c["slippage"]) * 2 * 100
        net_pct = g.mean() - rt
        weekly = net_pct * n_signals    # additive sum, not compounded
        print(f"  {scen:<14} ~{daily_trades:.0f} signals/day x  "
              f"{net_pct:+.4f}% net  ->  ~{weekly:+.1f}% over 7-day test")
    print(f"{'='*78}\n")


# -- multi-fee evaluation ------------------------------------------------------

def simulate_all_fee_scenarios(df: pd.DataFrame, horizon: int,
                                confidence_threshold: float = 0.0,
                                max_trades_per_day: int = None):
    """
    Run the strategy under every fee scenario in FEE_SCENARIOS.
    """
    out = {}
    for name, costs in FEE_SCENARIOS.items():
        per_side = costs["fee"] + costs["slippage"]
        tdf = simulate_strategy(df, horizon, tc=per_side,
                                confidence_threshold=confidence_threshold,
                                max_trades_per_day=max_trades_per_day)
        out[name] = {
            "trades":   tdf,
            "metrics":  backtest_metrics(tdf),
            "fee":      costs["fee"],
            "slippage": costs["slippage"],
        }
    return out


# -- confidence threshold sweep (diagnostic, taker fee only) -------------------

def confidence_sweep(df: pd.DataFrame, horizon: int, tc: float = 0.0005,
                     thresholds=None) -> pd.DataFrame:
    if thresholds is None:
        thresholds = [0.0, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

    rows = []
    for t in thresholds:
        tdf = simulate_strategy(df, horizon, tc, confidence_threshold=t)
        bm  = backtest_metrics(tdf)
        rows.append({"threshold": t, **bm})
    return pd.DataFrame(rows)


# -- plots ---------------------------------------------------------------------

def plot_equity_curves_multi_fee(scenarios_per_model: dict, model_name: str,
                                  title: str, save_path: str):
    """
    For a single model, plot the three fee-scenario equity curves on one axes.
    """
    fig, ax = plt.subplots(figsize=(11, 5))
    for scen, tdf in scenarios_per_model.items():
        if len(tdf) == 0:
            continue
        costs = FEE_SCENARIOS[scen]
        curve = (tdf["cumulative_return"].to_numpy()) * 100
        ax.plot(curve, label=(f"{scen}  fee={costs['fee']*100:+.4f}%  "
                              f"slip={costs['slippage']*100:.4f}% (per side)"),
                linewidth=1.6, color=SCENARIO_COLORS[scen])
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
    ax.set_xlabel("Trade #")
    ax.set_ylabel("Cumulative Return (%)")
    ax.set_title(f"{title}\n{model_name}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved {save_path}")


def plot_compare_models_per_scenario(curves_per_scenario: dict,
                                      title: str, save_path: str):
    """
    For each fee scenario, plot all-models equity curves side-by-side.
    """
    n_scen = len(curves_per_scenario)
    fig, axes = plt.subplots(1, n_scen, figsize=(6 * n_scen, 5), sharey=False)
    if n_scen == 1:
        axes = [axes]

    for ax, (scen, model_curves) in zip(axes, curves_per_scenario.items()):
        costs = FEE_SCENARIOS[scen]
        for m, curve in model_curves.items():
            if not curve:
                continue
            ax.plot(np.array(curve) * 100, label=m, linewidth=1.4)
        ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
        ax.set_xlabel("Trade #")
        ax.set_ylabel("Cumulative Return (%)")
        ax.set_title(f"{scen}\nfee={costs['fee']*100:+.4f}%  "
                     f"slip={costs['slippage']*100:.4f}% / side")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved {save_path}")


def plot_confidence_sweep(sweep_df: pd.DataFrame, model_name: str, save_path: str):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(sweep_df["threshold"], sweep_df["total_return"] * 100,
                 marker="o", color="steelblue")
    axes[0].axhline(0, color="red", linestyle="--", linewidth=0.8)
    axes[0].set_title("Total Return (%) vs Confidence Threshold")
    axes[0].set_xlabel("Confidence Threshold")
    axes[0].set_ylabel("Total Return (%)")

    axes[1].plot(sweep_df["threshold"], sweep_df["sharpe"],
                 marker="o", color="green")
    axes[1].axhline(0, color="red", linestyle="--", linewidth=0.8)
    axes[1].set_title("Sharpe vs Confidence Threshold")
    axes[1].set_xlabel("Confidence Threshold")
    axes[1].set_ylabel("Sharpe")

    ax2 = axes[1].twinx()
    ax2.plot(sweep_df["threshold"], sweep_df["n_trades"],
             marker="s", color="orange", linestyle="--", alpha=0.7)
    ax2.set_ylabel("# Trades", color="orange")

    axes[2].plot(sweep_df["threshold"], sweep_df["win_rate"] * 100,
                 marker="o", color="purple")
    axes[2].axhline(50, color="gray", linestyle="--", linewidth=0.8, label="50%")
    axes[2].set_title("Win Rate (%) vs Confidence Threshold")
    axes[2].set_xlabel("Confidence Threshold")
    axes[2].set_ylabel("Win Rate (%)")
    axes[2].legend()

    fig.suptitle(f"{model_name} -- Confidence Threshold Sweep (taker fee)", fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved {save_path}")


# -- pretty printers -----------------------------------------------------------

def _print_scenario_table(model_name: str, scenarios: dict, capital: float = 100_000):
    print(f"\n[{model_name}]   (Starting capital: ${capital:,.0f}  "
          f"position size = 100% per trade)")
    print(f"{'Scenario':<15} {'Fee/side':>9} {'Slip/side':>10} {'Trades':>7} "
          f"{'Tr/Day':>7} {'WinRate':>9} {'Ret%':>7} "
          f"{'P&L $':>11} {'Sharpe':>8} {'MaxDD%':>7}")
    print("-" * 100)
    for scen, payload in scenarios.items():
        m    = payload["metrics"]
        fee  = payload["fee"]
        slip = payload["slippage"]
        pnl_usd = capital * m["total_return"]
        print(f"{scen:<15} {fee*100:>+8.4f}% {slip*100:>+9.4f}% "
              f"{m['n_trades']:>7} {m['trades_per_day']:>7.0f} "
              f"{m['win_rate']:>9.3f} {m['total_return']*100:>7.2f} "
              f"{pnl_usd:>+11,.0f} {m['sharpe']:>8.3f} "
              f"{m['max_drawdown']*100:>7.2f}")


# -- main backtest function ----------------------------------------------------

# Operationally-realistic cap for a single retail trader on Binance Spot.
# 50 trades/day = roughly 1 trade every ~30 minutes during active hours.
# Strategies that need to trade more than this will hit market-impact and
# latency walls a backtest can't simulate.
REALISTIC_MAX_TRADES_PER_DAY = 50


def backtest(config: Config, symbol: str = "BTC", compare_all: bool = False,
             confidence_threshold: float = 0.0,
             max_trades_per_day: int = REALISTIC_MAX_TRADES_PER_DAY):
    if compare_all:
        print(f"\n{'='*78}")
        print(f"Backtest -- {symbol}  k{config.horizon_key}  "
              f"confidence>={confidence_threshold:.2f}  (3 fee scenarios)")
        print("Per-scenario costs (fee + slippage):")
        for s, c in FEE_SCENARIOS.items():
            total = (c['fee'] + c['slippage']) * 2 * 100
            print(f"  {s:<14} fee={c['fee']*100:+.4f}%/side  "
                  f"slip={c['slippage']*100:.4f}%/side  "
                  f"-> round-trip cost = {total:+.4f}%")
        print("Sharpe is PERIOD Sharpe over the 7-day test (mean/std of daily P&L), "
              "NOT annualised. SumRet% = additive sum of trade returns; "
              "interpret as % of fixed position size.")
        print(f"{'='*78}")

        model_list  = ["lr", "mlp", "cnn_lstm", "deeplob", "lob_transformer", "bigru_attn"]
        curves_per_scenario = {scen: {} for scen in FEE_SCENARIOS}
        all_metrics = {}

        for m in model_list:
            if not os.path.isfile(os.path.join(
                    config.results_dir, f"{m}_{symbol}_k{config.horizon_key}.json")):
                print(f"\n[{m}] not trained, skipping")
                continue
            cfg_m = Config(model_name=m, horizon_key=config.horizon_key)
            try:
                df = get_predictions_with_prices(cfg_m, symbol)
            except FileNotFoundError:
                print(f"\n[{m}] checkpoint missing, skipping")
                continue

            # diagnostic: per-trade economic breakdown
            diagnose_signal_economics(df, config.horizon_key,
                                       confidence_threshold=confidence_threshold)

            # Pass 1: unconstrained (HFT upper bound, theoretical)
            scen_unconstrained = simulate_all_fee_scenarios(
                df, config.horizon_key,
                confidence_threshold=confidence_threshold,
                max_trades_per_day=None
            )
            print(f"\n--- {m}: UNCONSTRAINED (HFT theoretical upper bound) ---")
            _print_scenario_table(m, scen_unconstrained)

            # Pass 2: realistic retail cap
            scen_realistic = simulate_all_fee_scenarios(
                df, config.horizon_key,
                confidence_threshold=confidence_threshold,
                max_trades_per_day=max_trades_per_day
            )
            print(f"\n--- {m}: REALISTIC (max {max_trades_per_day} trades/day) ---")
            _print_scenario_table(m, scen_realistic)

            all_metrics[m] = scen_realistic   # use realistic for downstream plots/tables

            scenarios = scen_realistic
            tdfs = {scen: payload["trades"] for scen, payload in scenarios.items()}
            plot_equity_curves_multi_fee(
                tdfs, model_name=m,
                title=(f"Equity Curves -- {symbol} k={config.horizon_key} "
                       f"conf>={confidence_threshold:.2f}  "
                       f"(<={max_trades_per_day} trades/day)"),
                save_path=os.path.join(
                    config.results_dir,
                    f"equity_realistic_{m}_{symbol}_k{config.horizon_key}.png"
                )
            )

            for scen, payload in scenarios.items():
                tdf = payload["trades"]
                curves_per_scenario[scen][m] = (
                    tdf["cumulative_return"].tolist() if len(tdf) > 0 else []
                )

        if any(curves_per_scenario.values()):
            plot_compare_models_per_scenario(
                curves_per_scenario,
                title=(f"Model Comparison -- {symbol} k={config.horizon_key} "
                       f"conf>={confidence_threshold:.2f}  "
                       f"<={max_trades_per_day} trades/day"),
                save_path=os.path.join(
                    config.results_dir,
                    f"equity_compare_{symbol}_k{config.horizon_key}.png"
                )
            )

        # Per-scenario summary tables (REALISTIC cap, max trades/day)
        print(f"\n\n{'#'*78}")
        print(f"# REALISTIC RESULTS  ({max_trades_per_day} trades/day cap, "
              f"~$100K capital)")
        print(f"# Each trade uses fixed position size; sum of % returns over 7-day test")
        print(f"{'#'*78}")
        for scen in FEE_SCENARIOS:
            costs = FEE_SCENARIOS[scen]
            rt    = (costs['fee'] + costs['slippage']) * 2 * 100
            print(f"\n{'-'*78}")
            print(f"Summary -- {scen} (fee={costs['fee']*100:+.4f}% + "
                  f"slip={costs['slippage']*100:.4f}% per side  "
                  f"=> round-trip {rt:+.4f}%)")
            print(f"{'-'*78}")
            print(f"{'Model':<22} {'Trades':>7} {'Tr/Day':>7} {'WinRate':>9} "
                  f"{'Ret%':>7} {'P&L $':>11} {'Sharpe':>8} {'MaxDD%':>7}")
            print("-" * 92)
            for m, scenarios in all_metrics.items():
                bm = scenarios[scen]["metrics"]
                pnl = 100_000 * bm["total_return"]
                print(f"{m:<22} {bm['n_trades']:>7} {bm['trades_per_day']:>7.0f} "
                      f"{bm['win_rate']:>9.3f} {bm['total_return']*100:>7.2f} "
                      f"{pnl:>+11,.0f} {bm['sharpe']:>8.3f} "
                      f"{bm['max_drawdown']*100:>7.2f}")

        return all_metrics

    # ─── single model ──────────────────────────────────────────────────────────
    print(f"\nBacktest: {config.model_name} | {symbol} | k{config.horizon_key} "
          f"| confidence>={confidence_threshold:.2f}")
    df = get_predictions_with_prices(config, symbol)

    diagnose_signal_economics(df, config.horizon_key,
                               confidence_threshold=confidence_threshold)

    print("\nConfidence threshold sweep (taker fee 0.05%/side):")
    sweep = confidence_sweep(df, config.horizon_key, tc=FEE_SCENARIOS["taker"])
    print(sweep[["threshold", "n_trades", "win_rate",
                 "total_return", "sharpe", "max_drawdown"]].to_string(index=False))

    pfx = f"{config.model_name}_{symbol}_k{config.horizon_key}"
    plot_confidence_sweep(sweep, config.model_name,
                          os.path.join(config.results_dir, f"conf_sweep_{pfx}.png"))

    scenarios = simulate_all_fee_scenarios(
        df, config.horizon_key, confidence_threshold=confidence_threshold
    )
    _print_scenario_table(config.model_name, scenarios)

    tdfs = {scen: payload["trades"] for scen, payload in scenarios.items()}
    plot_equity_curves_multi_fee(
        tdfs, model_name=config.model_name,
        title=(f"Equity Curves -- {symbol} k={config.horizon_key} "
               f"conf>={confidence_threshold:.2f}"),
        save_path=os.path.join(
            config.results_dir, f"equity_multifee_{pfx}.png"
        )
    )
    return scenarios


## 7. CLI Driver (`run.py`)

Master command-line entry point. All operations dispatched through `run.py preprocess / train / evaluate / backtest`.


In [ ]:
"""
Master CLI for the LOB Deep Learning project.

STEP 1 — Preprocess (run once per symbol, ~1-2 hours per file):
    python run.py preprocess --symbol BTC
    python run.py preprocess --symbol ETH
    python run.py preprocess --symbol BTC --debug    # fast test: 1 day only

STEP 2 — Train (repeat for each model / horizon combination):
    python run.py train --model lob_transformer --symbol BTC --horizon 10
    python run.py train --model deeplob         --symbol BTC --horizon 10
    python run.py train --model cnn_lstm        --symbol BTC --horizon 20
    python run.py train --model mlp             --symbol BTC --horizon 10
    python run.py train --model lr              --symbol BTC --horizon 10
    python run.py train --model lob_transformer --symbol BTC --horizon 10 --debug

STEP 3 — Evaluate:
    python run.py evaluate --model lob_transformer --symbol BTC --horizon 10
    python run.py evaluate --compare-all --symbol BTC --horizon 10

STEP 4 — Backtest:
    python run.py backtest --model lob_transformer --symbol BTC --horizon 10
    python run.py backtest --compare-all --symbol BTC --horizon 10

Quick end-to-end test (debug mode, ~5-10 min):
    python run.py preprocess --symbol BTC --debug
    python run.py train --model lob_transformer --symbol BTC --horizon 10 --debug
    python run.py evaluate --model lob_transformer --symbol BTC --horizon 10
    python run.py backtest --model lob_transformer --symbol BTC --horizon 10
"""

import argparse
import sys
import os

# ensure project root is on path
sys.path.insert(0, os.path.dirname(__file__))

from config import Config


def main():
    parser = argparse.ArgumentParser(
        description="LOB Deep Learning — main pipeline",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    sub = parser.add_subparsers(dest="command", required=True)

    # ── preprocess ──────────────────────────────────────────────────────────
    p_pre = sub.add_parser("preprocess", help="JSON -> Parquet preprocessing")
    p_pre.add_argument("--symbol", choices=["BTC", "ETH", "both"], default="BTC")
    p_pre.add_argument("--debug", action="store_true",
                       help="Process only ~5000 snapshots (≈83 min of data)")

    # ── train ────────────────────────────────────────────────────────────────
    p_tr = sub.add_parser("train", help="Train a model")
    p_tr.add_argument("--model", default="lob_transformer",
                      choices=["lr", "mlp", "cnn_lstm", "deeplob", "lob_transformer", "bigru_attn"])
    p_tr.add_argument("--symbol",  default="BTC", choices=["BTC", "ETH"])
    p_tr.add_argument("--horizon", default=10, type=int, choices=[10, 20, 50])
    p_tr.add_argument("--epochs",  default=None, type=int)
    p_tr.add_argument("--debug",   action="store_true",
                      help="Tiny dataset, 3 epochs — for quick testing")

    # ── evaluate ─────────────────────────────────────────────────────────────
    p_ev = sub.add_parser("evaluate", help="Evaluate model(s)")
    p_ev.add_argument("--model", default="lob_transformer",
                      choices=["lr", "mlp", "cnn_lstm", "deeplob", "lob_transformer", "bigru_attn"])
    p_ev.add_argument("--symbol",      default="BTC", choices=["BTC", "ETH"])
    p_ev.add_argument("--horizon",     default=10, type=int, choices=[10, 20, 50])
    p_ev.add_argument("--compare-all", action="store_true",
                      help="Print comparison table for all trained models")

    # ── backtest ─────────────────────────────────────────────────────────────
    p_bt = sub.add_parser("backtest", help="Simulate trading strategy")
    p_bt.add_argument("--model", default="lob_transformer",
                      choices=["lr", "mlp", "cnn_lstm", "deeplob", "lob_transformer", "bigru_attn"])
    p_bt.add_argument("--symbol",      default="BTC", choices=["BTC", "ETH"])
    p_bt.add_argument("--horizon",     default=10, type=int, choices=[10, 20, 50])
    p_bt.add_argument("--compare-all", action="store_true",
                      help="Compare all trained models in backtest")
    p_bt.add_argument("--confidence", default=0.0, type=float,
                      help="Min softmax confidence to trade (default 0.0 = all signals)")
    p_bt.add_argument("--max-trades", default=50, type=int,
                      help="Max trades per day (operational realism cap, default 50)")

    args = parser.parse_args()

    # ── dispatch ─────────────────────────────────────────────────────────────

    if args.command == "preprocess":
        from data.preprocess import preprocess_symbol
        cfg = Config(debug=args.debug)
        symbols = ["BTC", "ETH"] if args.symbol == "both" else [args.symbol]
        for sym in symbols:
            preprocess_symbol(sym, cfg, debug=args.debug)

    elif args.command == "train":
        from train import train
        debug = args.debug
        extra = {}
        if args.epochs:
            extra["epochs"] = args.epochs
        if debug:
            extra["epochs"] = 3
            extra["debug"]  = True
        cfg = Config(model_name=args.model, horizon_key=args.horizon, **extra)
        train(cfg, symbol=args.symbol)

    elif args.command == "evaluate":
        from evaluate import evaluate
        cfg = Config(model_name=args.model, horizon_key=args.horizon)
        evaluate(cfg, symbol=args.symbol, compare_all=args.compare_all)

    elif args.command == "backtest":
        from backtest import backtest
        cfg = Config(model_name=args.model, horizon_key=args.horizon)
        backtest(cfg, symbol=args.symbol, compare_all=args.compare_all,
                 confidence_threshold=args.confidence,
                 max_trades_per_day=args.max_trades)


if __name__ == "__main__":
    main()


## 8. Utilities (`utils.py`)

Shared metrics, class-weight computation, and logging helpers.


In [ ]:
"""Shared metrics, class-weight computation, and logging helpers."""

import os
import json
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    matthews_corrcoef, classification_report,
)


CLASS_NAMES = ["DOWN", "STATIONARY", "UP"]


def compute_class_weights(labels: np.ndarray, n_classes: int = 3,
                           device: str = "cpu") -> torch.Tensor:
    """Inverse-frequency weights for imbalanced LOB labels."""
    counts = np.bincount(labels, minlength=n_classes).astype(float)
    counts = np.where(counts == 0, 1.0, counts)
    weights = counts.sum() / (n_classes * counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Return dict of standard classification metrics."""
    return {
        "accuracy":  float(accuracy_score(y_true, y_pred)),
        "f1_macro":  float(f1_score(y_true, y_pred, average="macro",  zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred)),
        "mcc":         float(matthews_corrcoef(y_true, y_pred)),
        "f1_per_class": f1_score(y_true, y_pred, average=None,
                                  labels=[0, 1, 2], zero_division=0).tolist(),
    }


def print_metrics(metrics: dict, prefix: str = ""):
    tag = f"[{prefix}] " if prefix else ""
    print(f"{tag}Acc={metrics['accuracy']:.4f}  "
          f"F1-macro={metrics['f1_macro']:.4f}  "
          f"Kappa={metrics['cohen_kappa']:.4f}  "
          f"MCC={metrics['mcc']:.4f}")
    classes = {0: "DOWN", 1: "STAT", 2: "UP"}
    f1s = metrics["f1_per_class"]
    print(f"{tag}F1: " + "  ".join(f"{classes[i]}={f1s[i]:.3f}" for i in range(3)))


def save_results(results: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to {path}")


def load_results(path: str) -> dict:
    with open(path) as f:
        return json.load(f)


## 9. Innovation Code

Source for the two original contributions: animated attention + FGSM robustness.

### 9a. FGSM adversarial attack


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

def fgsm_attack(model, X, y, epsilon, model_name):
    """Generate adversarial inputs via Fast Gradient Sign Method.

    X_adv = X + epsilon * sign(grad_X loss)

    cuDNN LSTM backward requires train mode; we set then restore.
    """
    was_training = model.training
    model.train()
    X_adv = X.clone().detach().requires_grad_(True)
    if model_name == 'lob_transformer':
        out = model(X_adv)
        logits = out[0] if isinstance(out, tuple) else out
    else:
        logits = model(X_adv)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    perturbation = epsilon * X_adv.grad.sign()
    if not was_training:
        model.eval()
    return (X_adv + perturbation).detach()


### 9b. Animated attention - frame collection


In [ ]:
import numpy as np
import torch

@torch.no_grad()
def collect_attention_frames(model, test_loader, device, n_frames=200):
    """Collect per-sample level-attention weights for n_frames samples."""
    attn_frames, predictions, true_labels = [], [], []
    for batch in test_loader:
        X, y_dir, _ = batch
        B, T = X.shape[0], X.shape[1]
        X = X.to(device)
        dir_logits, _ = model(X)
        attn = model.get_level_attention()
        if attn is None:
            continue
        attn_np = attn.detach().cpu().numpy()
        if attn_np.ndim == 4:
            per_sample = attn_np[:, -1]
        elif attn_np.ndim == 3 and attn_np.shape[0] == B * T:
            per_sample = attn_np.reshape(B, T, attn_np.shape[1], attn_np.shape[2])[:, -1]
        else:
            per_sample = attn_np
        for i in range(B):
            attn_frames.append(per_sample[i])
            predictions.append(int(dir_logits.argmax(dim=1)[i].item()))
            true_labels.append(int(y_dir[i].item()))
            if len(attn_frames) >= n_frames:
                break
        if len(attn_frames) >= n_frames:
            break
    return np.array(attn_frames), np.array(predictions), np.array(true_labels)


---

## How to actually run this code

From a terminal in the project folder:

```bash
# 1. Preprocess raw JSON -> daily Parquet
python run.py preprocess --symbol BTC

# 2. Train all 6 models at all 3 horizons
for m in lr mlp cnn_lstm deeplob lob_transformer bigru_attn; do
  for k in 10 20 50; do
    python run.py train --model $m --symbol BTC --horizon $k --epochs 10
  done
done

# 3. Classification metrics + plots
python run.py evaluate --compare-all --symbol BTC --horizon 10

# 4. Multi-fee backtest with realistic constraints
python run.py backtest --compare-all --symbol BTC --horizon 10 \
                       --confidence 0.75 --max-trades 50
```

Total runtime on RTX 5080: ~4 hours.
